# EDA del histórico de Pulso TransMi

Este cuaderno perfila las 12 estaciones y el histórico de 51.840 observaciones (frecuencia de 15 minutos). Los timestamps se normalizan a UTC para el análisis, aunque el API publica el corte con zona horaria de Bogotá.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data' / 'starter'
obs = pd.read_csv(DATA / 'observations.csv', dtype={'station_id': 'string'})
obs['ts'] = pd.to_datetime(obs['observed_at'], utc=True)
obs['hour'] = obs['ts'].dt.hour
obs['weekday'] = obs['ts'].dt.dayofweek
obs['weekday_name'] = obs['ts'].dt.day_name()
obs = obs.sort_values(['station_id', 'ts']).reset_index(drop=True)
print(f'Filas: {len(obs):,} | estaciones: {obs.station_id.nunique()} | desde: {obs.ts.min()} | hasta: {obs.ts.max()}')

Filas: 51,840 | estaciones: 12 | desde: 2026-07-26 05:00:00+00:00 | hasta: 2026-09-09 04:45:00+00:00


## Perfil por estación

In [2]:
station_profile = (obs.groupby('station_id')['demand'].agg(
    observations='size', mean='mean', median='median', std='std',
    minimum='min', maximum='max', p01=lambda s: s.quantile(.01),
    p99=lambda s: s.quantile(.99),
).round(2))
display(station_profile)

,observations,mean,median,std,minimum,maximum,p01,p99
station_id,,,,,,,,
02300,4320,293.97,158.0,279.29,29,1472,43.00,1201.24
03000,4320,258.43,153.0,216.83,48,1252,65.00,948.62
05000,4320,342.06,201.0,288.81,59,1944,86.00,1284.81
05100,4320,591.06,392.0,391.06,159,2065,194.00,1604.81
06000,4320,510.94,340.5,343.01,127,1789,163.19,1450.00
06111,4320,238.89,207.0,169.95,14,835,25.00,677.81
07105,4320,272.59,213.5,217.44,42,1313,53.00,964.67
07107,4320,281.03,248.0,198.73,17,982,30.00,802.81
07111,4320,683.72,452.0,459.36,159,2284,224.00,1903.00


## Estacionalidad horaria

In [3]:
hourly = (obs.groupby('hour')['demand'].agg(mean='mean', median='median', std='std', observations='size').round(2))
display(hourly)
display(obs.pivot_table(index='hour', columns='station_id', values='demand', aggfunc='mean').round(1))

,mean,median,std,observations
hour,,,,
0,534.51,476.5,275.73,2160
1,383.03,314.5,253.13,2160
2,280.84,202.0,225.57,2160
3,213.93,142.0,163.98,2160
4,172.20,126.0,112.04,2160
5,151.06,116.0,93.37,2160
6,139.55,107.5,89.14,2160
7,137.83,103.0,92.14,2160
8,148.85,109.0,101.93,2160


station_id,02300,03000,05000,05100,06000,06111,07105,07107,07111,09000,09122,10009
hour,,,,,,,,,,,,
0,409.3,419.2,466.0,827.6,824.9,278.0,744.4,311.7,759.8,273.8,237.8,861.6
1,175.7,302.1,298.1,475.6,497.7,240.2,715.3,231.9,456.8,170.2,114.6,918.1
2,113.0,186.0,189.8,324.1,310.9,160.3,563.8,144.5,340.8,111.0,92.3,833.5
3,102.1,125.4,143.2,281.3,246.1,97.7,375.7,91.6,321.9,88.0,88.1,606.1
4,103.0,102.9,129.2,273.8,234.8,66.9,224.3,72.3,314.3,82.5,86.2,376.0
5,104.4,95.4,125.5,277.2,241.1,59.9,134.2,67.3,318.5,79.5,86.8,223.1
6,101.6,96.2,125.8,273.7,236.0,56.3,95.7,65.9,316.1,80.7,86.6,140.1
7,102.3,96.2,126.1,280.6,237.5,56.5,81.4,67.5,324.3,83.0,89.9,108.6
8,101.1,102.6,157.3,293.3,247.5,58.9,77.0,75.6,369.6,117.9,89.8,95.8


## Estacionalidad por día de semana

In [4]:
weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekday = (obs.groupby(['weekday', 'weekday_name'])['demand']
           .agg(mean='mean', median='median', std='std', observations='size')
           .reset_index().sort_values('weekday').set_index('weekday_name'))
display(weekday[ ['mean', 'median', 'std', 'observations'] ].reindex(weekday_order).round(2))
display(obs.pivot_table(index='weekday_name', columns='hour', values='demand', aggfunc='mean').reindex(weekday_order).round(1))

,mean,median,std,observations
weekday_name,,,,
Monday,374.09,269.0,340.08,8064
Tuesday,383.87,288.0,333.73,8064
Wednesday,380.11,286.0,329.41,7152
Thursday,379.28,283.0,331.42,6912
Friday,376.01,285.0,323.70,6912
Saturday,304.71,226.0,275.29,6912
Sunday,297.13,215.0,279.80,7824


hour,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
weekday_name,,,,,,,,,,,,,,,,,,,,,
Monday,477.5,360.1,271.3,202.4,160.1,159.0,150.4,146.7,161.3,230.3,...,404.9,289.9,268.6,276.1,275.0,289.3,373.0,580.3,745.5,735.6
Tuesday,560.0,401.6,293.2,222.2,180.2,160.6,148.6,148.3,158.0,227.8,...,394.7,291.7,277.0,282.7,285.0,293.2,383.1,580.7,752.6,729.1
Wednesday,552.5,398.2,283.5,219.3,176.8,159.6,145.9,149.9,162.2,230.9,...,399.5,292.8,266.6,278.2,273.4,286.0,392.7,575.6,758.3,750.7
Thursday,564.7,385.6,283.4,217.1,177.3,156.1,146.3,146.0,158.3,225.2,...,390.2,288.8,268.9,279.3,281.7,296.0,387.5,579.5,765.5,748.1
Friday,573.4,401.1,285.9,217.2,177.5,159.5,152.8,147.2,156.2,231.6,...,388.4,290.6,269.1,273.9,275.0,285.1,366.4,579.4,740.4,718.7
Saturday,552.4,386.7,287.2,223.4,178.4,133.0,117.6,114.0,124.9,178.9,...,276.5,208.4,199.0,205.0,206.4,216.6,291.1,440.6,579.9,573.2
Sunday,463.4,346.1,260.5,195.4,155.1,130.1,115.8,113.6,121.9,178.3,...,277.7,216.4,204.4,213.1,215.0,225.8,294.4,444.1,571.1,578.2


## Faltantes y regularidad temporal

In [5]:
expected_times = pd.date_range(obs.ts.min(), obs.ts.max(), freq='15min', tz='UTC')
expected = pd.MultiIndex.from_product([obs.station_id.unique(), expected_times], names=['station_id', 'ts'])
observed = pd.MultiIndex.from_frame(obs[['station_id', 'ts']])
missing = expected.difference(observed)
missing_by_station = missing.to_frame(index=False).groupby('station_id').size().rename('missing_rows')
missing_report = pd.DataFrame({'expected_rows': expected.size // obs.station_id.nunique(), 'observed_rows': obs.groupby('station_id').size()}).join(missing_by_station, how='left').fillna(0).astype(int)
print(f'Faltantes totales: {len(missing):,}')
display(missing_report)
gaps = obs.sort_values(['station_id', 'ts']).groupby('station_id')['ts'].diff().value_counts().head(10)
display(gaps.rename('gap_count').to_frame())

Faltantes totales: 0


,expected_rows,observed_rows,missing_rows
station_id,,,
02300,4320,4320,0
03000,4320,4320,0
05000,4320,4320,0
05100,4320,4320,0
06000,4320,4320,0
06111,4320,4320,0
07105,4320,4320,0
07107,4320,4320,0
07111,4320,4320,0


,gap_count
ts,
0 days 00:15:00,51828


## Outliers por estación (regla IQR)

In [6]:
def outlier_summary(group):
    q1, q3 = group['demand'].quantile([.25, .75])
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    mask = (group['demand'] < low) | (group['demand'] > high)
    return pd.Series({'q1': q1, 'q3': q3, 'lower_bound': low, 'upper_bound': high, 'outliers': mask.sum(), 'outlier_rate': mask.mean()})
outliers = obs.groupby('station_id', group_keys=False).apply(outlier_summary, include_groups=False).round(4)
display(outliers)
display(obs.join(outliers[['lower_bound', 'upper_bound']], on='station_id').query('demand < lower_bound or demand > upper_bound')[['station_id', 'ts', 'demand']].head(20))

,q1,q3,lower_bound,upper_bound,outliers,outlier_rate
station_id,,,,,,
02300,111.00,399.00,-321.000,831.000,298.0,0.0690
03000,103.00,368.25,-294.875,766.125,204.0,0.0472
05000,136.00,484.00,-386.000,1006.000,201.0,0.0465
05100,293.00,861.25,-559.375,1713.625,19.0,0.0044
06000,250.00,742.00,-488.000,1480.000,30.0,0.0069
06111,87.00,354.00,-313.500,754.500,12.0,0.0028
07105,98.00,357.00,-290.500,745.500,191.0,0.0442
07107,101.75,413.00,-365.125,879.875,8.0,0.0019
07111,336.00,992.00,-648.000,1976.000,28.0,0.0065


,station_id,ts,demand
130,02300,2026-07-27 13:30:00+00:00,978
131,02300,2026-07-27 13:45:00+00:00,877
132,02300,2026-07-27 14:00:00+00:00,901
162,02300,2026-07-27 21:30:00+00:00,893
164,02300,2026-07-27 22:00:00+00:00,924
165,02300,2026-07-27 22:15:00+00:00,1269
166,02300,2026-07-27 22:30:00+00:00,1213
167,02300,2026-07-27 22:45:00+00:00,934
168,02300,2026-07-27 23:00:00+00:00,1129
170,02300,2026-07-27 23:30:00+00:00,1100


## Cinco hipótesis para modelar

1. La demanda presenta una estacionalidad intradía consistente, con niveles diferenciados entre franjas valle y punta.
2. El patrón de un mismo slot de la semana anterior será una señal fuerte, especialmente para estaciones con comportamiento semanal estable.
3. Las estaciones con mayor demanda tendrán mayor variabilidad absoluta, por lo que conviene comparar errores relativos y no solo errores absolutos.
4. Los outliers se concentrarán en franjas y estaciones específicas, posiblemente asociados a cambios de régimen, eventos o errores de medición; no deben eliminarse automáticamente.
5. Un modelo que combine el último valor, la estacionalidad semanal y una media móvil podrá superar a cada baseline aislado en horizontes de 15 a 60 minutos.